### ЗАДАЧА: Панель обработки проблемных доставок по паттерну `MVC`

Логистическая команда обрабатывает кейсы по проблемным доставкам.
Для каждого кейса нужно хранить:
- `case_id` — идентификатор кейса;
- `shipment_id` — идентификатор отправления;
- `courier` — служба доставки;
- `delay_hours` — задержка в часах;
- `status` — текущий статус;
- `operator` — сотрудник, который ведет кейс;
- `compensation_ready` — подготовлена ли компенсация;
- `resolution` — финальное решение.

Статусы кейса:
- `new`
- `investigating`
- `customer_contacted`
- `resolved`
- `escalated`

Бизнес-правила:
- нельзя создать кейс с уже существующим `case_id`;
- нельзя начать расследование без назначенного `operator`;
- перевести кейс в `customer_contacted` можно только из `investigating`;
- подготовить компенсацию можно только из `customer_contacted`;
- завершить кейс как `resolved` можно только из `customer_contacted`;
- финальный кейс (`resolved`, `escalated`) нельзя менять дальше;
- при `resolved` нужно записать `resolution` и оставить `compensation_ready` как есть;
- эскалировать можно только из `investigating` или `customer_contacted`.

In [ ]:
from dataclasses import dataclass

initial_cases = [
    ("DL-100", "SHP-11", "FastBox", 6),
    ("DL-101", "SHP-12", "QuickShip", 18),
]

actions = [
    ("show",),
    ("investigate", "DL-100"),
    ("assign", "DL-100", "Olga"),
    ("investigate", "DL-100"),
    ("contact", "DL-100"),
    ("resolve", "DL-100", "refund_sent"),
    ("create", "DL-102", "SHP-13", "CityRun", 30),
    ("assign", "DL-102", "Max"),
    ("investigate", "DL-102"),
    ("escalate", "DL-102"),
    ("contact", "DL-102"),
    ("show",),
]

@dataclass
class DeliveryCase:
    case_id: str
    shipment_id: str
    courier: str
    delay_hours: int
    status: str = "new"
    operator: str | None = None
    compensation_ready: bool = False
    resolution: str | None = None

class DeliveryModel:
    def __init__(self):
        # TODO 1: создайте self.cases как пустой словарь
        self.cases = {}

    def add_case(self, case_id: str, shipment_id: str, courier: str, delay_hours: int) -> None:
        # TODO 2: если case_id уже существует, вызовите ValueError('Case already exists')
        if case_id in self.cases:
            raise ValueError("Case already exists")
        # TODO 3: создайте DeliveryCase и сохраните в self.cases
        self.cases[case_id] = DeliveryCase(case_id, shipment_id, courier, delay_hours)

    def assign_operator(self, case_id: str, operator: str) -> None:
        # TODO 4: если кейс не найден, вызовите ValueError("Case not found")
        if case_id not in self.cases:
            raise ValueError("Case not found")
        # TODO 5: назначьте оператора
        self.cases[case_id].operator = operator

    def start_investigation(self, case_id: str) -> None:
        # TODO 6: если кейс не найден, вызовите ValueError("Case not found")
        if case_id not in self.cases:
            raise ValueError("Case not found")
        case = self.cases[case_id]
        # TODO 7: если оператор не назначен, вызовите ValueError("Operator is required")
        if case.operator is None:
            raise ValueError("Operator is required")
        # TODO 8: если статус финальный (resolved/escalated), вызовите ValueError("Final cases cannot be changed")
        if case.status in ("resolved", "escalated"):
            raise ValueError("Final cases cannot be changed")
        # TODO 9: установите status = "investigating"
        case.status = "investigating"

    def contact_customer(self, case_id: str) -> None:
        # TODO 10: если кейс не найден, вызовите ValueError("Case not found")
        if case_id not in self.cases:
            raise ValueError("Case not found")
        case = self.cases[case_id]
        # TODO 11: если статус не "investigating", вызовите ValueError("Can only contact customer from investigating")
        if case.status != "investigating":
            raise ValueError("Can only contact customer from investigating")
        # TODO 12: установите status = "customer_contacted"
        case.status = "customer_contacted"

    def prepare_compensation(self, case_id: str) -> None:
        # TODO 13: если кейс не найден, вызовите ValueError("Case not found")
        if case_id not in self.cases:
            raise ValueError("Case not found")
        case = self.cases[case_id]
        # TODO 14: если статус не "customer_contacted", вызовите ValueError("Can only prepare compensation from customer_contacted")
        if case.status != "customer_contacted":
            raise ValueError("Can only prepare compensation from customer_contacted")
        # TODO 15: установите compensation_ready = True
        case.compensation_ready = True

    def resolve_case(self, case_id: str, resolution: str) -> None:
        # TODO 16: если кейс не найден, вызовите ValueError("Case not found")
        if case_id not in self.cases:
            raise ValueError("Case not found")
        case = self.cases[case_id]
        # TODO 17: если статус не "customer_contacted", вызовите ValueError("Can only resolve from customer_contacted")
        if case.status != "customer_contacted":
            raise ValueError("Can only resolve from customer_contacted")
        # TODO 18: установите status = "resolved" и resolution
        case.status = "resolved"
        case.resolution = resolution

    def escalate_case(self, case_id: str) -> None:
        # TODO 19: если кейс не найден, вызовите ValueError("Case not found")
        if case_id not in self.cases:
            raise ValueError("Case not found")
        case = self.cases[case_id]
        # TODO 20: если статус финальный (resolved) или не в (investigating, customer_contacted), вызовите ValueError("Can only escalate from investigating or customer_contacted")
        if case.status == "resolved" or case.status not in ("investigating", "customer_contacted"):
            raise ValueError("Can only escalate from investigating or customer_contacted")
        # TODO 21: установите status = "escalated"
        case.status = "escalated"

    def list_cases(self) -> list[str]:
        # TODO 22: верните список строк формата:
        #   'case_id | shipment_id | courier | delay_hours | status | operator | compensation_ready | resolution'
        rows = []
        for case in self.cases.values():
            operator = case.operator if case.operator else "None"
            resolution = case.resolution if case.resolution else "None"
            row = (
                f"{case.case_id} | {case.shipment_id} | {case.courier} | "
                f"{case.delay_hours} | {case.status} | {operator} | "
                f"{case.compensation_ready} | {resolution}"
            )
            rows.append(row)
        return rows

class DeliveryView:
    @staticmethod
    def render_cases(rows: list[str]) -> None:
        # TODO 23: если rows пустой, выведите 'Кейсов нет'
        if not rows:
            print('Кейсов нет')
            return
        # TODO 24: иначе выведите заголовок 'Кейсы проблемных доставок:' и все строки
        print('Кейсы проблемных доставок:')
        for row in rows:
            print(row)

    @staticmethod
    def render_success(message: str) -> None:
        # TODO 25: выведите сообщение успеха
        print(f"Успех: {message}")

    @staticmethod
    def render_error(message: str) -> None:
        # TODO 26: выведите сообщение ошибки
        print(f"Ошибка: {message}")

class DeliveryController:
    def __init__(self, model: DeliveryModel, view: DeliveryView):
        # TODO 27: сохраните model и view
        self.model = model
        self.view = view

    def create_case(self, case_id: str, shipment_id: str, courier: str, delay_hours: int) -> None:
        # TODO 28: через try/except вызовите model.add_case(...)
        try:
            self.model.add_case(case_id, shipment_id, courier, delay_hours)
            # TODO 29: покажите success или error
            self.view.render_success(f"Кейс {case_id} создан")
        except ValueError as error:
            self.view.render_error(str(error))

    def assign_operator(self, case_id: str, operator: str) -> None:
        # TODO 30: через try/except вызовите model.assign_operator(...)
        try:
            self.model.assign_operator(case_id, operator)
            # TODO 31: покажите success или error
            self.view.render_success(f"Оператор {operator} назначен на кейс {case_id}")
        except ValueError as error:
            self.view.render_error(str(error))

    def start_investigation(self, case_id: str) -> None:
        # TODO 32: через try/except вызовите model.start_investigation(...)
        try:
            self.model.start_investigation(case_id)
            # TODO 33: покажите success или error
            self.view.render_success(f"Кейс {case_id} отправлен на расследование")
        except ValueError as error:
            self.view.render_error(str(error))

    def contact_customer(self, case_id: str) -> None:
        # TODO 34: через try/except вызовите model.contact_customer(...)
        try:
            self.model.contact_customer(case_id)
            # TODO 35: покажите success или error
            self.view.render_success(f"Клиент по кейсу {case_id} уведомлён")
        except ValueError as error:
            self.view.render_error(str(error))

    def prepare_compensation(self, case_id: str) -> None:
        # TODO 36: через try/except вызовите model.prepare_compensation(...)
        try:
            self.model.prepare_compensation(case_id)
            # TODO 37: покажите success или error
            self.view.render_success(f"Компенсация подготовлена для кейса {case_id}")
        except ValueError as error:
            self.view.render_error(str(error))

    def resolve_case(self, case_id: str, resolution: str) -> None:
        # TODO 38: через try/except вызовите model.resolve_case(...)
        try:
            self.model.resolve_case(case_id, resolution)
            # TODO 39: покажите success или error
            self.view.render_success(f"Кейс {case_id} завершён с резолюцией '{resolution}'")
        except ValueError as error:
            self.view.render_error(str(error))

    def escalate_case(self, case_id: str) -> None:
        # TODO 40: через try/except вызовите model.escalate_case(...)
        try:
            self.model.escalate_case(case_id)
            # TODO 41: покажите success или error
            self.view.render_success(f"Кейс {case_id} эскалатирован")
        except ValueError as error:
            self.view.render_error(str(error))

    def show_cases(self) -> None:
        # TODO 42: получите строки из model.list_cases() и передайте их во view.render_cases(...)
        cases = self.model.list_cases()
        self.view.render_cases(cases)

model = DeliveryModel()
view = DeliveryView()
controller = DeliveryController(model, view)

# TODO 43: загрузите начальные кейсы из initial_cases в model
# Подсказка: пройдите по initial_cases циклом for
# и вызовите model.add_case(case_id, shipment_id, courier, delay_hours)
for case_id, shipment_id, courier, delay_hours in initial_cases:
    try:
        model.add_case(case_id, shipment_id, courier, delay_hours)
        view.render_success(f"Начальный кейс {case_id} загружен")
    except ValueError as e:
        view.render_error(f"Ошибка загрузки начального кейса {case_id}: {str(e)}")

# TODO 44: обработайте все действия из actions
for action in actions:
    # TODO 45: если action[0] == 'show', вызовите controller.show_cases()
    if action[0] == "show":
        controller.show_cases()
    # TODO 46: если action[0] == 'create', распакуйте case_id, shipment_id, courier, delay_hours
    # и вызовите controller.create_case(...)
    elif action[0] == "create":
        _, case_id, shipment_id, courier, delay_hours = action
        controller.create_case(case_id, shipment_id, courier, delay_hours)
    # TODO 47: если action[0] == 'assign', распакуйте case_id и operator
    # и вызовите controller.assign_operator(...)
    elif action[0] == "assign":
        _, case_id, operator = action
        controller.assign_operator(case_id, operator)
    # TODO 48: если action[0] == 'investigate', распакуйте case_id
    # и вызовите controller.start_investigation(...)
    elif action[0] == "investigate":
        _, case_id = action
        controller.start_investigation(case_id)
    # TODO 49: если action[0] == 'contact', распакуйте case_id
    # и вызовите controller.contact_customer(...)
    elif action[0] == "contact":
        _, case_id = action
        controller.contact_customer(case_id)
    # TODO 50: если action[0] == 'prepare', распакуйте case_id
    # и вызовите controller.prepare_compensation(...)
    elif action[0] == "prepare":
        _, case_id = action
        controller.prepare_compensation(case_id)
    # TODO 51: если action[0] == 'resolve', распакуйте case_id и resolution
    # и вызовите controller.resolve_case(...)
    elif action[0] == "resolve":
        _, case_id, resolution = action
        controller.resolve_case(case_id, resolution)
    # TODO 52: если action[0] == 'escalate', распакуйте case_id
    # и вызовите controller.escalate_case(...)
    elif action[0] == "escalate":
        _, case_id = action
        controller.escalate_case(case_id)

print()
print("Финальное состояние:")
# TODO 53: покажите финальное состояние через controller.show_cases()
controller.show_cases()


In [ ]:
# Delivery cases:
# DL-100 | SHP-11 | FastBox | 6 | new | None | False | None
# DL-101 | SHP-12 | QuickShip | 18 | new | None | False | None
# ERROR: Operator is required
# SUCCESS: Operator assigned to DL-100
# SUCCESS: Case DL-100 moved to investigation
# SUCCESS: Customer contacted for DL-100
# SUCCESS: Case DL-100 resolved
# SUCCESS: Case DL-102 created
# SUCCESS: Operator assigned to DL-102
# SUCCESS: Case DL-102 moved to investigation
# SUCCESS: Case DL-102 escalated
# ERROR: Case is not investigating
# Delivery cases:
# DL-100 | SHP-11 | FastBox | 6 | resolved | Olga | False | refund_sent
# DL-101 | SHP-12 | QuickShip | 18 | new | None | False | None
# DL-102 | SHP-13 | CityRun | 30 | escalated | Max | False | escalated
#
# Финальное состояние:
# Delivery cases:
# DL-100 | SHP-11 | FastBox | 6 | resolved | Olga | False | refund_sent
# DL-101 | SHP-12 | QuickShip | 18 | new | None | False | None
# DL-102 | SHP-13 | CityRun | 30 | escalated | Max | False | escalated